# `source_final.ipynb` — finalized pipeline (incremental)

This notebook is the **production** version of the sources pipeline.

Rule: we only copy stages into here once they were validated in `sources_test.ipynb`.

**Status**
- ✅ Stage B (Chapter Blueprint) finalized: `coverage_v1` (wins vs baseline)
- ✅ Stage C scoring finalized: `w_embed_max=0.0`, `w_embed=0.7`, `cite_weight=0.08`
- ✅ Stage C.3 finalized: LLM rerank **within Stage C top-50** + calibrated scoring + deterministic tie-break (model: `gpt-5-nano`)
- ✅ Stage D enabled: TF-IDF MMR selection for diverse top-20 (can be disabled)
- ✅ Stage A/C.2 included: OpenAlex + Semantic Scholar fetch; embeddings + TF-IDF scoring


In [1]:
from __future__ import annotations

import os
import json
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field

# OpenAI / Agents SDK (used for schema-validated JSON output)
from openai import OpenAI
from agents import Agent, Runner, ModelSettings


# -----------------------------
# Repo/workspace paths
# -----------------------------
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "package.json").exists() and (parent / "README.md").exists():
            return parent
    return p


REPO_ROOT = find_repo_root()
SOURCES_WORKSPACE_DIR = REPO_ROOT / "sources_workspace"


# -----------------------------
# Minimal .env loader (notebook convenience)
# -----------------------------
def load_dotenv_minimal(path: str = ".env") -> None:
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and (k not in os.environ):
            os.environ[k] = v


load_dotenv_minimal(str(REPO_ROOT / ".env"))

if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError("Missing OPENAI_API_KEY. Add it to your environment or .env file.")

client = OpenAI()  # validates credentials on first real request

# -----------------------------
# Output dirs (cache)
# -----------------------------
OUT_DIR = SOURCES_WORKSPACE_DIR / "final_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STAGEB_DIR = OUT_DIR / "stageB_blueprints"
STAGEB_DIR.mkdir(parents=True, exist_ok=True)

# Additional pipeline dirs
STAGEA_DIR = OUT_DIR / "stageA"
STAGEA_DIR.mkdir(parents=True, exist_ok=True)
STAGEC_DIR = OUT_DIR / "stageC"
STAGEC_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Run id (used for output file names; caches are stable across runs)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# -----------------------------
# Finalized decision from Stage B A/B testing
# -----------------------------
STAGEB_FINAL_VARIANT = "coverage_v1"  # ✅ finalized winner

# Model pricing (USD per 1M tokens) — keep updated if pricing changes.
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    "text-embedding-3-small": {"input": 0.02, "cached": 0.0, "output": 0.0},
}

# Stage B model (only a few calls per chapter, but we still cost-track)
BLUEPRINT_MODEL = "gpt-5-mini"

# Rebuild controls
FORCE_REBUILD_BLUEPRINTS = False

# -----------------------------
# Chapter specs (edit/replace these to use the pipeline on new chapters)
# -----------------------------
CHAPTERS: List[Dict[str, Any]] = [
{
  "chapter_id": "zero_trust_architecture",
  "title": "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken",
  "original_text_language": "de",
  "original_text": (
    "Ziel ist eine präzise, technische Fundierung von Zero Trust Architecture (ZTA) für Unternehmens-IT (On‑Prem, Cloud, Hybrid), "
    "um später eine konkrete ZTA‑Einführung bewerten/planen zu können.\n\n"
    "(1) Begriffsdefinition und Abgrenzung: Zero Trust vs. klassische Perimeter‑Sicherheit, 'assume breach', identitätszentrierte Sicherheit. "
    "Abgrenzung zu reinem IAM‑/MFA‑Thema und zu allgemeinen 'Trust'-Konzepten außerhalb der IT.\n"
    "(2) Kernprinzipien: explizit verifizieren (Identity/Device/Context), Least Privilege, kontinuierliche Autorisierung, Segmentierung/Mikrosegmentierung, "
    "Least‑Privilege Netzwerkzugriff, Policy‑basierte Zugriffskontrolle.\n"
    "(3) Referenzarchitekturen und Bausteine: Control plane vs. data plane; Policy Decision Point/Policy Enforcement Point; "
    "Identität (Users/Workloads), Device Posture, Netzwerksegmentierung, Access Proxies, Service‑to‑Service AuthN/AuthZ, Secrets/Keys‑Handling (nur als Baustein).\n"
    "(4) Telemetrie & kontinuierliche Bewertung: Logging/Monitoring, Security Signals, kontinuierliche Risiko‑Bewertung, "
    "Detektion von Kompromittierung, Incident‑Response‑Integration.\n"
    "(5) Migration in Legacy‑Umgebungen: typische Einführungsstrategien/Patterns, Priorisierung kritischer Assets, "
    "Stolpersteine (Komplexität, Latenz, Policy‑Sprawl), und wie man ZTA iterativ einführt.\n"
    "(6) Bewertungskriterien: messbare Outcomes (z.B. Reduktion lateral movement‑Risiko, Policy‑Coverage, Mean Time to Detect/Respond), "
    "Validierungsansätze, Grenzen und häufige Fehlinterpretationen.\n\n"
    "Einschlüsse: Standards/Frameworks (z.B. NIST‑artige Referenzen), systematische Übersichtsarbeiten, Architektur‑ und Evaluationspapiere, "
    "empirische Studien zu Wirksamkeit/Trade‑offs.\n"
    "Ausschlüsse: keine Produktvergleiche/Tool‑Buyers‑Guides; keine Pen‑Testing‑How‑Tos; keine allgemeinen Kryptographie‑Einführungen; "
    "kein Fokus auf Datenschutzrecht/Compliance (nur falls direkt für ZTA‑Design relevant); keine Blockchain/Dezentralisierung als Hauptthema."
  ),
}

]

print("Config OK")
print("Stage B finalized variant:", STAGEB_FINAL_VARIANT)
print("Chapters:", [c["chapter_id"] for c in CHAPTERS])


Config OK
Stage B finalized variant: coverage_v1
Chapters: ['zero_trust_architecture']


## Stage B (finalized): Chapter Blueprint generation (`coverage_v1`)

This is the only Stage B variant that goes into `source_final.ipynb`.


In [2]:
# -----------------------------
# Stage B: build blueprints (cached)
# -----------------------------

class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")

    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]

    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]

    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None

    scoring_guidance: str
    notes: Optional[str] = None


def price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    """Token totals + estimated cost using MODEL_PRICES_USD_PER_1M."""

    prices = price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    # Fallback: aggregated totals
    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


BASE_BLUEPRINT_INSTRUCTIONS = (
    "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
    "Return ONLY the structured output fields (no extra text).\n\n"
    "Constraints:\n"
    "- language must be 'en'\n"
    "- scope_statement: 1 sentence, <= 30 words\n"
    "- must_cover: 4\u20138 bullets, each <= 16 words\n"
    "- should_cover: 3\u20138 bullets, each <= 16 words\n"
    "- must_avoid: 3\u20138 bullets, each <= 16 words\n"
    "- main_query: <= 18 words\n"
    "- facet_queries: 8\u201314 items, each <= 14 words\n"
    "- keywords: 20\u201345 items\n"
    "- key_concepts: 10\u201322 items\n"
    "- preferred_source_types: 2\u20136 items\n"
    "- negative_query_terms: 0\u201312 items derived from must_avoid (soft negatives)\n"
    "- scoring_guidance: <= 80 words\n"
    "- Do NOT contradict yourself: if something is in must_avoid, do not emphasize it in keywords/facets.\n"
    "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
)

COVERAGE_V1_INSTRUCTIONS = BASE_BLUEPRINT_INSTRUCTIONS + (
    "\nAdditional requirements (coverage_v1):\n"
    "- facet_queries must be semantically diverse (avoid near-duplicates).\n"
    "- Ensure each must_cover bullet is explicitly targeted by at least one facet_query.\n"
)

BLUEPRINT_INSTRUCTIONS = COVERAGE_V1_INSTRUCTIONS

blueprint_agent = Agent(
    name="Chapter Blueprint Builder (coverage_v1)",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=BLUEPRINT_INSTRUCTIONS,
    output_type=ChapterBlueprint,
)


async def get_or_create_blueprint(chapter: dict) -> tuple[ChapterBlueprint, dict]:
    out_path = STAGEB_DIR / f"{chapter['chapter_id']}.json"

    if (not FORCE_REBUILD_BLUEPRINTS) and out_path.exists():
        bp_dict = json.loads(out_path.read_text(encoding="utf-8"))
        bp_obj = ChapterBlueprint.model_validate(bp_dict)
        return bp_obj, {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}

    prompt = (
        "Create a ChapterBlueprint for academic literature retrieval.\n"
        "Return ONLY the structured output fields required by the schema.\n\n"
        "CHAPTER_SPEC_JSON:\n"
        + json.dumps(chapter, ensure_ascii=False, indent=2)
    )

    res = await Runner.run(blueprint_agent, prompt)
    bp_obj = res.final_output
    bp = bp_obj.model_dump()
    bp["_meta"] = {
        "blueprint_variant": STAGEB_FINAL_VARIANT,
        "model": BLUEPRINT_MODEL,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

    usage = res.context_wrapper.usage
    return bp_obj, cost_from_usage(usage, model=BLUEPRINT_MODEL)


blueprints: Dict[str, ChapterBlueprint] = {}
bp_totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
for ch in CHAPTERS:
    bp, u = await get_or_create_blueprint(ch)
    blueprints[ch["chapter_id"]] = bp
    for k in bp_totals:
        bp_totals[k] += u.get(k, 0)

print("Stage B blueprints ready")
print("Stage B cost summary:", bp_totals)
for cid, bp in blueprints.items():
    print("-", cid, "| facets:", len(bp.facet_queries or []), "| main:", bp.main_query)


Stage B blueprints ready
Stage B cost summary: {'requests': 1, 'input_tokens': 1131, 'cached_input_tokens': 0, 'output_tokens': 3187, 'cost_usd': 0.00665675}
- zero_trust_architecture | facets: 12 | main: technical foundations of Zero Trust Architecture for enterprise networks


## Stage A (production): API retrieval (OpenAlex + Semantic Scholar)

Build the Stage A corpus per chapter (cached to disk).


In [3]:
# -----------------------------
# Stage A: API retrieval (OpenAlex + Semantic Scholar)
# -----------------------------

# This cell writes per-chapter StageA corpora to:
#   final_pipeline/stageA/<chapter_id>/<sig>/stageA_combined.csv

import os
import re
import json
import time
import random
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import List, Optional, Dict, Any, Iterable

import requests
import numpy as np
import pandas as pd

OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY", "").strip()
if not OPENALEX_API_KEY:
    print("Warning: OPENALEX_API_KEY not set; continuing without it (may be slower / more rate limits).")

S2_API_KEY = os.getenv("SEMANTICSCHOLAR_API_KEY", "").strip()
S2_MIN_INTERVAL = 1.0  # seconds (rate limit: 1 rps)
_S2_LAST_TS = 0.0
if S2_API_KEY:
    print("Semantic Scholar API key detected (SEMANTICSCHOLAR_API_KEY). Using authenticated requests (1 req/sec).")
else:
    print("Note: SEMANTICSCHOLAR_API_KEY not set; Semantic Scholar will use long retry/backoff without a key.")

# -----------------------------
# Stage A knobs
# -----------------------------
FETCH_FORCE = False
RUN_OPENALEX = True
RUN_SEMANTIC_SCHOLAR = True

FETCH_MAX_QUERIES_PER_CHAPTER = 15

# OpenAlex
OA_BASE_URL = "https://api.openalex.org/works"
OA_PER_PAGE = 100
OA_MAX_WORKS_PER_QUERY = 300
OA_TIMEOUT_SEC = 30

# Semantic Scholar
S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_SEARCH_URL = f"{S2_BASE}/paper/search"
S2_BATCH_URL  = f"{S2_BASE}/paper/batch"

S2_LIMIT = 100
S2_MAX_PAGES_PER_QUERY = 1
S2_FETCH_ABSTRACTS_VIA_BATCH = True
S2_BATCH_SIZE = 200
S2_CACHE_ENABLED = True
S2_CACHE_TTL_DAYS = 30
S2_VERBOSE = False

# Robust retries/backoff (important without S2_API_KEY)
S2_TIMEOUT_SEC = 30
S2_REQUEST_MAX_RETRIES = 200
S2_REQUEST_MAX_SECONDS = 3600
S2_BACKOFF_INITIAL_SEC = 2.0
S2_BACKOFF_MAX_SEC = 300.0
S2_BACKOFF_JITTER = 0.25
S2_SUCCESS_SLEEP_SEC = 0.2

# Shared Semantic Scholar cache (re-used across notebooks)
S2_CACHE_DIR = SOURCES_WORKSPACE_DIR / "eval_dataset" / "fetch" / "s2_cache"
S2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Stage A: fetch OpenAlex + Semantic Scholar per chapter
# and build per-chapter StageA CSVs
# -----------------------------

# Output files:
# - eval_dataset/datasets/<dataset_tag>/fetch/openalex_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/fetch/semantic_scholar_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/stageA/stageA_combined_oa_s2_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/stageA/stageA_all_chapters.csv

def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        x = str(x).strip()
        if not x:
            continue
        k = x.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out

def query_list_for_chapter(bp) -> List[str]:
    # Accept both dict blueprints and ChapterBlueprint objects
    if hasattr(bp, "main_query"):
        main = bp.main_query
        facets = list(bp.facet_queries or [])
    else:
        main = (bp.get("main_query", "") if isinstance(bp, dict) else "")
        facets = list((bp.get("facet_queries") or []) if isinstance(bp, dict) else [])
    qs = [main] + facets
    qs = dedupe_preserve_order(qs)
    return qs[:FETCH_MAX_QUERIES_PER_CHAPTER]

# ----------------------------
# OpenAlex
# ----------------------------

def abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    if not inv:
        return None

    pairs: List[tuple[int, str]] = []
    for word, positions in inv.items():
        if not positions:
            continue
        for p in positions:
            if isinstance(p, int):
                pairs.append((p, word))

    if not pairs:
        return None

    pairs.sort(key=lambda x: x[0])
    max_pos = pairs[-1][0]
    words = [""] * (max_pos + 1)
    for pos, w in pairs:
        if 0 <= pos <= max_pos:
            words[pos] = w

    text = " ".join(w for w in words if w).strip()
    return text or None

def venue_from_primary_location(work: Dict[str, Any]) -> Optional[str]:
    pl = work.get("primary_location") or {}
    src = pl.get("source") or {}
    return src.get("display_name")

def first_n_authors(work: Dict[str, Any], n: int = 6) -> str:
    authors = []
    for a in (work.get("authorships") or [])[:n]:
        name = ((a.get("author") or {}).get("display_name"))
        if name:
            authors.append(name)
    return "; ".join(authors)

def oa_get(params: Dict[str, Any], max_retries: int = 6) -> dict:
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        r = requests.get(OA_BASE_URL, params=params, timeout=OA_TIMEOUT_SEC)
        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
            time.sleep(backoff)
            backoff *= 2
            continue
        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
        return r.json()
    raise RuntimeError("OpenAlex retry loop exhausted")

def fetch_openalex_query(q: str, max_works: Optional[int]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    cursor = "*"
    select = (
        "id,display_name,publication_year,type,doi,cited_by_count,"
        "authorships,primary_location,abstract_inverted_index"
    )

    while cursor:
        params: Dict[str, Any] = {
            "search": q,
            "per-page": OA_PER_PAGE,
            "cursor": cursor,
            "select": select,
        }
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY

        data = oa_get(params)

        for w in data.get("results", []) or []:
            rows.append({
                "query": q,
                "title": w.get("display_name"),
                "year": w.get("publication_year"),
                "type": w.get("type"),
                "venue": venue_from_primary_location(w),
                "cited_by": w.get("cited_by_count"),
                "authors(first6)": first_n_authors(w, n=6),
                "doi": w.get("doi"),
                "openalex_id": w.get("id"),
                "abstract": abstract_from_inverted_index(w.get("abstract_inverted_index")),
            })

            if max_works is not None and len(rows) >= max_works:
                return rows

        cursor = (data.get("meta") or {}).get("next_cursor")
        if not cursor:
            break

    return rows

def fetch_openalex_for_chapter(chapter_id: str, queries: List[str]) -> pd.DataFrame:
    all_rows: List[Dict[str, Any]] = []
    for qi, q in enumerate(queries, start=1):
        print(f"[OpenAlex:{chapter_id}] ({qi}/{len(queries)}) {q}")
        all_rows.extend(fetch_openalex_query(q, max_works=OA_MAX_WORKS_PER_QUERY))

    df_oa = pd.DataFrame(all_rows)
    if df_oa.empty:
        return df_oa

    df_oa.insert(0, "chapter_id", chapter_id)
    df_oa = df_oa.drop_duplicates(subset=["chapter_id", "query", "openalex_id"], keep="first").reset_index(drop=True)
    return df_oa

# ----------------------------
# Semantic Scholar
# ----------------------------

# Shared Semantic Scholar cache across dataset versions (avoids repeated API calls)
S2_CACHE_DIR = SOURCES_WORKSPACE_DIR / "eval_dataset" / "fetch" / "s2_cache"
S2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _now_utc() -> datetime:
    return datetime.now(timezone.utc)

def _cache_key(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> str:
    blob = {"m": method.upper(), "u": url, "p": params or {}, "b": body or {}}
    s = json.dumps(blob, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(s).hexdigest()

def s2_cache_get(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> Optional[Any]:
    if not S2_CACHE_ENABLED:
        return None
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        created = datetime.fromisoformat(payload["created"])
        if _now_utc() - created > timedelta(days=S2_CACHE_TTL_DAYS):
            return None
        return payload["data"]
    except Exception:
        return None

def s2_cache_set(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]], data: Any) -> None:
    if not S2_CACHE_ENABLED:
        return
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    payload = {"created": _now_utc().isoformat(), "data": data}
    path.write_text(json.dumps(payload), encoding="utf-8")

def _parse_retry_after(resp: requests.Response) -> Optional[float]:
    ra = resp.headers.get("Retry-After")
    if not ra:
        return None
    try:
        return float(ra)
    except ValueError:
        return None

def s2_request(
    session: requests.Session,
    method: str,
    url: str,
    params: Optional[Dict[str, Any]] = None,
    body: Optional[Dict[str, Any]] = None,
    max_retries: Optional[int] = None,
    max_elapsed_seconds: Optional[float] = None,
) -> Any:
    """Semantic Scholar request with caching + long retry/backoff.

    Without an API key, S2 can rate-limit aggressively (429). We therefore:
    - retry for a long time (default up to ~1 hour per request)
    - respect Retry-After when present
    - use exponential backoff + jitter
    - treat network exceptions as retryable
    """
    global _S2_LAST_TS

    cached = s2_cache_get(method, url, params, body)
    if cached is not None:
        return cached

    max_retries = int(max_retries if max_retries is not None else S2_REQUEST_MAX_RETRIES)
    max_elapsed_seconds = float(max_elapsed_seconds if max_elapsed_seconds is not None else S2_REQUEST_MAX_SECONDS)

    start = time.monotonic()
    backoff = float(S2_BACKOFF_INITIAL_SEC)
    last_status: Any = None
    last_err: Optional[str] = None

    attempt = 0
    while True:
        attempt += 1
        # rate limit: 1 request/second
        now = time.time()
        wait = S2_MIN_INTERVAL - (now - _S2_LAST_TS)
        if wait > 0:
            time.sleep(wait)
        _S2_LAST_TS = time.time()

        resp: Optional[requests.Response]
        try:
            resp = session.request(method, url, params=params, json=body, timeout=S2_TIMEOUT_SEC)
            last_status = resp.status_code
            last_err = None
        except Exception as e:
            resp = None
            last_status = "exception"
            last_err = repr(e)

        if resp is not None and resp.status_code == 200:
            try:
                data = resp.json()
            except Exception as e:
                last_status = "json_error"
                last_err = repr(e)
            else:
                s2_cache_set(method, url, params, body, data)
                return data

        ra = None
        if resp is None:
            retryable = True
        elif resp.status_code in (429, 500, 502, 503, 504):
            retryable = True
            ra = _parse_retry_after(resp)
        elif resp.status_code in (408,):
            retryable = True
        else:
            raise RuntimeError(f"S2 error {resp.status_code} | body: {resp.text[:500]}")

        elapsed = time.monotonic() - start
        if attempt >= max_retries or elapsed >= max_elapsed_seconds:
            detail = f"last_status={last_status}"
            if last_err:
                detail += f" last_err={last_err}"
            raise RuntimeError(
                f"S2 retry budget exhausted after {elapsed:.0f}s and {attempt} attempts: {method} {url} ({detail})"
            )

        wait = float(backoff)
        if ra is not None:
            wait = max(wait, float(ra))
        wait = min(float(S2_BACKOFF_MAX_SEC), wait)
        if S2_BACKOFF_JITTER:
            jitter = 1.0 + random.uniform(-float(S2_BACKOFF_JITTER), float(S2_BACKOFF_JITTER))
            wait = max(0.0, wait * jitter)

        # Minimum sleep to avoid hammering the API
        wait = max(1.0, wait)

        remaining = max_elapsed_seconds - elapsed
        wait = min(wait, max(0.0, remaining))

        if S2_VERBOSE:
            print(
                f"[S2] status={last_status} retry in {wait:.1f}s | attempt {attempt}/{max_retries} | elapsed {elapsed:.0f}s"
            )
        time.sleep(wait)
        backoff = min(float(S2_BACKOFF_MAX_SEC), backoff * 2)

def clean_query(q: str) -> str:
    q = q.replace("-", " ")
    q = re.sub(r"\s+", " ", q).strip()
    return q

def chunks(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def fetch_s2_for_chapter(chapter_id: str, queries: List[str], out_csv: Optional[Path] = None) -> pd.DataFrame:
    """Fetch Semantic Scholar results with:
    - long retry/backoff (works without API key)
    - incremental CSV checkpointing (never lose progress)
    - resume support via a small progress JSON
    - optional abstract fetching with a resumable abstract cache
    """
    SEARCH_FIELDS = "paperId,title,year,authors,venue,citationCount,externalIds,url"

    DETAIL_FIELDS = "paperId,abstract"

    base_cols = [
        "chapter_id",
        "query",
        "paperId",
        "title",
        "year",
        "venue",
        "citationCount",
        "authors(first6)",
        "doi",
        "s2_url",
    ]
    csv_cols = base_cols + ["abstract"]

    session = requests.Session()
    session.headers.update({"User-Agent": "instantpaper-eval/1.0"})
    if S2_API_KEY:
        session.headers.update({"x-api-key": S2_API_KEY})

    # --- Resume state (progress + seen keys) ---
    progress_path: Optional[Path] = None
    abstracts_cache_path: Optional[Path] = None
    completed_queries: set[str] = set()
    seen: set[tuple[str, str]] = set()

    queries_sha12 = hashlib.sha1(json.dumps(list(queries), ensure_ascii=False).encode("utf-8")).hexdigest()[:12]

    if out_csv is not None:
        out_csv = Path(out_csv)
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        progress_path = out_csv.with_suffix(".progress.json")
        abstracts_cache_path = out_csv.with_suffix(".abstracts.json")

        if out_csv.exists():
            try:
                existing = pd.read_csv(out_csv)
                if "abstract" not in existing.columns:
                    existing["abstract"] = np.nan
                    tmp_csv = out_csv.with_suffix(".tmp")
                    existing.to_csv(tmp_csv, index=False, encoding="utf-8")
                    tmp_csv.replace(out_csv)
                for q0, pid0 in zip(existing.get("query", []), existing.get("paperId", [])):
                    if pd.isna(q0) or pd.isna(pid0):
                        continue
                    seen.add((str(q0), str(pid0)))
            except Exception:
                # If the file is corrupted/unreadable, we still keep appending new rows.
                pass

        if progress_path.exists():
            try:
                payload = json.loads(progress_path.read_text(encoding="utf-8"))
                if payload.get("queries_sha1_12") == queries_sha12:
                    completed_queries = set(payload.get("completed_queries", []) or [])
            except Exception:
                pass

    def save_progress() -> None:
        if progress_path is None:
            return
        payload = {
            "chapter_id": chapter_id,
            "queries_sha1_12": queries_sha12,
            "completed_queries": sorted(completed_queries),
            "n_completed": int(len(completed_queries)),
            "n_total": int(len(queries)),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        progress_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    def append_rows(rows_new: List[Dict[str, Any]]) -> None:
        if not rows_new:
            return
        if out_csv is None:
            rows.extend(rows_new)
            return
        df_new = pd.DataFrame(rows_new)
        for c in csv_cols:
            if c not in df_new.columns:
                df_new[c] = np.nan
        df_new = df_new[csv_cols]
        header = (not out_csv.exists()) or (out_csv.stat().st_size == 0)
        df_new.to_csv(out_csv, mode="a", index=False, header=header, encoding="utf-8")

    # If we are not writing to disk, keep rows in memory.
    rows: List[Dict[str, Any]] = []

    # --- Search loop (incremental checkpointing) ---
    for qi, q in enumerate(queries, start=1):
        if q in completed_queries:
            continue

        q2 = clean_query(q)
        print(f"[S2:{chapter_id}] ({qi}/{len(queries)}) {q2}")

        offset = 0
        for _page in range(1, S2_MAX_PAGES_PER_QUERY + 1):
            params = {
                "query": q2,
                "fields": SEARCH_FIELDS,
                "limit": S2_LIMIT,
                "offset": offset,
            }
            data = s2_request(session, "GET", S2_SEARCH_URL, params=params, body=None)
            time.sleep(S2_SUCCESS_SLEEP_SEC)

            papers = data.get("data", []) or []

            batch_rows: List[Dict[str, Any]] = []
            for p in papers:
                pid = p.get("paperId")
                if not pid:
                    continue
                key = (str(q), str(pid))
                if key in seen:
                    continue
                seen.add(key)

                authors = [a.get("name") for a in (p.get("authors") or [])[:6] if a.get("name")]
                ext = p.get("externalIds") or {}
                batch_rows.append(
                    {
                        "chapter_id": chapter_id,
                        "query": q,
                        "paperId": pid,
                        "title": p.get("title"),
                        "year": p.get("year"),
                        "venue": p.get("venue"),
                        "citationCount": p.get("citationCount"),
                        "authors(first6)": "; ".join(authors),
                        "doi": ext.get("DOI"),
                        "s2_url": p.get("url"),
                    }
                )

            append_rows(batch_rows)

            if len(papers) < S2_LIMIT:
                break
            offset += S2_LIMIT

        completed_queries.add(q)
        save_progress()

    # Load current results
    if out_csv is not None and out_csv.exists():
        df_s2 = pd.read_csv(out_csv)
    else:
        df_s2 = pd.DataFrame(rows)

    if df_s2.empty:
        return df_s2

    df_s2 = df_s2.drop_duplicates(subset=["chapter_id", "query", "paperId"]).reset_index(drop=True)

    # --- Abstract fetching (resumable) ---
    if S2_FETCH_ABSTRACTS_VIA_BATCH:
        abstracts: Dict[str, Optional[str]] = {}

        # Load existing abstract cache
        if abstracts_cache_path is not None and abstracts_cache_path.exists():
            try:
                abstracts.update(json.loads(abstracts_cache_path.read_text(encoding="utf-8")))
            except Exception:
                pass

        # Also trust already-present abstracts in CSV
        if "abstract" in df_s2.columns:
            for pid, abs_ in zip(df_s2.get("paperId", []), df_s2.get("abstract", [])):
                if pd.isna(pid) or pd.isna(abs_):
                    continue
                abstracts[str(pid)] = str(abs_)

        unique_ids = [str(pid) for pid in df_s2["paperId"].dropna().unique().tolist() if pid]
        missing_ids = [pid for pid in unique_ids if pid not in abstracts]

        if missing_ids:
            print(f"[S2:{chapter_id}] fetching abstracts via batch | missing={len(missing_ids)}/{len(unique_ids)}")

        for ids in chunks(missing_ids, S2_BATCH_SIZE):
            params = {"fields": DETAIL_FIELDS}
            body = {"ids": ids}
            batch = s2_request(session, "POST", S2_BATCH_URL, params=params, body=body)
            time.sleep(S2_SUCCESS_SLEEP_SEC)

            if isinstance(batch, list):
                it = batch
            else:
                it = (batch.get("data", []) or [])

            for p in it:
                pid = p.get("paperId")
                if pid:
                    abstracts[str(pid)] = p.get("abstract")

            # Persist abstract cache after each chunk (checkpoint)
            if abstracts_cache_path is not None:
                tmp = abstracts_cache_path.with_suffix(".tmp")
                tmp.write_text(json.dumps(abstracts, ensure_ascii=False), encoding="utf-8")
                tmp.replace(abstracts_cache_path)

        df_s2["abstract"] = df_s2["paperId"].astype(str).map(abstracts)

    # Always persist the latest full CSV (including abstracts if enabled)
    if out_csv is not None:
        tmp_csv = out_csv.with_suffix(".tmp")
        df_s2.to_csv(tmp_csv, index=False, encoding="utf-8")
        tmp_csv.replace(out_csv)

    return df_s2

# ----------------------------
# Stage A merge (OpenAlex + S2) per chapter
# ----------------------------

def standardize_openalex(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "openalex"
    d["source_id"] = d.get("openalex_id")
    d["citation_count"] = d.get("cited_by")
    for col in ["abstract", "doi", "title", "year", "venue", "type", "authors(first6)", "query", "chapter_id", "openalex_id"]:
        if col not in d.columns:
            d[col] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "title", "year", "venue", "type",
        "authors(first6)", "doi", "citation_count", "abstract", "openalex_id"
    ]]

def standardize_s2(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "semantic_scholar"
    d["source_id"] = d.get("paperId")
    d["citation_count"] = d.get("citationCount")
    for col in ["abstract", "doi", "title", "year", "venue", "authors(first6)", "query", "paperId", "s2_url", "chapter_id"]:
        if col not in d.columns:
            d[col] = np.nan
    d["type"] = np.nan
    d["openalex_id"] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "title", "year", "venue", "type",
        "authors(first6)", "doi", "citation_count", "abstract", "paperId", "s2_url"
    ]]

def normalize_doi(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"^https?://(dx\\.)?doi\\.org/", "", x)
    x = re.sub(r"^doi:\\s*", "", x)
    x = x.strip()
    return x or None

def normalize_title(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"[\\u2010-\\u2015]", "-", x)
    x = re.sub(r"[^a-z0-9\\s]", " ", x)
    x = re.sub(r"\\s+", " ", x).strip()
    return x or None

def longest_text(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    return max(vals, key=len)

def most_common_or_longest(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vc = pd.Series(vals).value_counts()
    if len(vc) and vc.iloc[0] >= 2:
        return vc.index[0]
    return max(vals, key=len)

def first_nonnull(series):
    for v in series:
        if pd.notna(v) and v not in ("", None):
            return v
    return None

def within_source_key(df_in: pd.DataFrame) -> pd.Series:
    k = []
    for _, r in df_in.iterrows():
        if r.get("doi_norm"):
            k.append(f"doi:{r['doi_norm']}")
        elif pd.notna(r.get("source_id")):
            k.append(f"id:{r['source']}:{r['source_id']}")
        else:
            y = int(r["year"]) if pd.notna(r.get("year")) else ""
            k.append(f"ty:{r['title_norm']}|{y}")
    return pd.Series(k, index=df_in.index)

def cross_source_merge_key(r: pd.Series) -> str:
    if isinstance(r.get("doi_norm"), str) and r.get("doi_norm"):
        return f"doi:{r['doi_norm']}"
    if pd.notna(r.get("year")):
        return f"ty:{r['title_norm']}|{int(round(float(r['year'])))}"
    return f"t:{r['title_norm']}"

def build_stagea(df_oa_raw: pd.DataFrame, df_s2_raw: pd.DataFrame, chapter_id: str) -> pd.DataFrame:
    oa_std = standardize_openalex(df_oa_raw)
    s2_std = standardize_s2(df_s2_raw)

    combined_raw = pd.concat([oa_std, s2_std], ignore_index=True)
    stageA = combined_raw.copy()

    stageA["doi_norm"] = stageA["doi"].map(normalize_doi)
    stageA["title_norm"] = stageA["title"].map(normalize_title)
    stageA["citation_count"] = pd.to_numeric(stageA["citation_count"], errors="coerce")

    stageA = stageA[stageA["title_norm"].notna()].reset_index(drop=True)
    stageA["within_key"] = within_source_key(stageA)

    agg_map = {
        "chapter_id": lambda s: s.iloc[0],
        "source": lambda s: s.iloc[0],
        "source_id": first_nonnull,
        "title": most_common_or_longest,
        "title_norm": lambda s: s.iloc[0],
        "year": lambda s: pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan,
        "venue": most_common_or_longest,
        "type": most_common_or_longest,
        "authors(first6)": most_common_or_longest,
        "doi": first_nonnull,
        "doi_norm": first_nonnull,
        "citation_count": lambda s: pd.to_numeric(s, errors="coerce").max(),
        "abstract": longest_text,
        "query": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "openalex_id": first_nonnull,
        "paperId": first_nonnull,
        "s2_url": first_nonnull,
    }

    stageA_dedup = (
        stageA
        .groupby(["source", "within_key"], as_index=False)
        .agg(agg_map)
        .drop(columns=["within_key"])
    )

    stageA_dedup["merge_key"] = stageA_dedup.apply(cross_source_merge_key, axis=1)

    def merge_sources(group: pd.DataFrame) -> dict:
        sources = sorted(set(group["source"].dropna().tolist()))
        source_ids = {src: group.loc[group["source"] == src, "source_id"].dropna().astype(str).unique().tolist() for src in sources}

        return {
            "chapter_id": chapter_id,
            "merge_key": group["merge_key"].iloc[0],
            "sources": "; ".join(sources),
            "source_count": len(sources),
            "source_ids": str(source_ids),
            "title": most_common_or_longest(group["title"]),
            "year": pd.to_numeric(group["year"], errors="coerce").dropna().median() if group["year"].notna().any() else np.nan,
            "venue": most_common_or_longest(group["venue"]),
            "type": most_common_or_longest(group["type"]),
            "authors(first6)": most_common_or_longest(group["authors(first6)"]),
            "doi": first_nonnull(group["doi"]),
            "doi_norm": first_nonnull(group["doi_norm"]),
            "citation_count_max": pd.to_numeric(group["citation_count"], errors="coerce").max(),
            "abstract": longest_text(group["abstract"]),
            "queries": "; ".join(sorted(set(
                q for q in group["query"].dropna().tolist()
                if isinstance(q, str) and q.strip()
            ))),
            "openalex_id": first_nonnull(group.get("openalex_id", pd.Series([], dtype=object))),
            "paperId": first_nonnull(group.get("paperId", pd.Series([], dtype=object))),
            "s2_url": first_nonnull(group.get("s2_url", pd.Series([], dtype=object))),
        }

    merged_records = [merge_sources(g) for _, g in stageA_dedup.groupby("merge_key")]
    df_stageA = pd.DataFrame(merged_records)

    df_stageA["merge_kind"] = df_stageA["merge_key"].str.split(":", n=1).str[0]
    df_stageA["has_abstract"] = df_stageA["abstract"].notna() & (df_stageA["abstract"].astype(str).str.len() > 50)
    df_stageA = df_stageA.sort_values(by="citation_count_max", ascending=False, na_position="last").reset_index(drop=True)

    return df_stageA

# ----------------------------

# -----------------------------
# Run Stage A for all chapters
# -----------------------------

stageA_by_chapter: Dict[str, pd.DataFrame] = {}

for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    queries = query_list_for_chapter(bp)

    sig_obj = {
        "queries": queries,
        "OA_MAX_WORKS_PER_QUERY": OA_MAX_WORKS_PER_QUERY,
        "S2_LIMIT": S2_LIMIT,
        "S2_MAX_PAGES_PER_QUERY": S2_MAX_PAGES_PER_QUERY,
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True).encode("utf-8")).hexdigest()[:12]

    ch_dir = STAGEA_DIR / cid / sig
    ch_dir.mkdir(parents=True, exist_ok=True)
    oa_csv = ch_dir / "openalex.csv"
    s2_csv = ch_dir / "semantic_scholar.csv"
    stagea_csv = ch_dir / "stageA_combined.csv"

    print(f"\n=== Stage A for chapter: {cid} | queries={len(queries)} | sig={sig} ===")

    # OpenAlex
    if RUN_OPENALEX:
        if oa_csv.exists() and not FETCH_FORCE:
            df_oa = pd.read_csv(oa_csv)
            print(f"[OpenAlex:{cid}] cached rows: {len(df_oa)}")
        else:
            df_oa = fetch_openalex_for_chapter(cid, queries)
            df_oa.to_csv(oa_csv, index=False, encoding="utf-8")
            print(f"[OpenAlex:{cid}] saved: {oa_csv} | rows={len(df_oa)}")
    else:
        df_oa = pd.DataFrame()

    # Semantic Scholar
    if RUN_SEMANTIC_SCHOLAR:
        # Always call the fetcher so it can resume if the CSV exists but is incomplete.
        df_s2 = fetch_s2_for_chapter(cid, queries, out_csv=s2_csv)
        print(f"[S2:{cid}] ready: {s2_csv} | rows={len(df_s2)}")
    else:
        df_s2 = pd.DataFrame()

    if stagea_csv.exists() and not FETCH_FORCE:
        df_stageA = pd.read_csv(stagea_csv)
        print(f"[StageA:{cid}] cached rows: {len(df_stageA)}")
    else:
        df_stageA = build_stagea(df_oa, df_s2, chapter_id=cid)
        df_stageA.to_csv(stagea_csv, index=False, encoding="utf-8")
        print(f"[StageA:{cid}] saved: {stagea_csv} | rows={len(df_stageA)}")

    stageA_by_chapter[cid] = df_stageA

print("\nStage A complete")
for cid, df0 in stageA_by_chapter.items():
    print("-", cid, "| rows:", len(df0), "| sources:", df0.get("sources", pd.Series(dtype=str)).value_counts().to_dict())


Semantic Scholar API key detected (SEMANTICSCHOLAR_API_KEY). Using authenticated requests (1 req/sec).

=== Stage A for chapter: zero_trust_architecture | queries=13 | sig=fa4c38fa4185 ===
[OpenAlex:zero_trust_architecture] (1/13) technical foundations of Zero Trust Architecture for enterprise networks
[OpenAlex:zero_trust_architecture] (2/13) definitions Zero Trust vs perimeter security literature
[OpenAlex:zero_trust_architecture] (3/13) assume breach and identity‑centered security papers
[OpenAlex:zero_trust_architecture] (4/13) Zero Trust core principles least privilege continuous authorization
[OpenAlex:zero_trust_architecture] (5/13) microsegmentation and least‑privileged network access studies
[OpenAlex:zero_trust_architecture] (6/13) control plane and data plane reference architectures
[OpenAlex:zero_trust_architecture] (7/13) Policy Decision Point Policy Enforcement Point implementations
[OpenAlex:zero_trust_architecture] (8/13) device posture attestation and workload identity

## Stage C (finalized): scoring weights

Chosen via `sources_test.ipynb` Stage C grid search (`stageC_grid_v1_best`).

**Final weights**
- `w_embed_max = 0.0` (use `score_embed_mean_top3` only)
- `w_embed = 0.7`
- `cite_weight = 0.08`


In [4]:
import numpy as np
import pandas as pd

# Finalized Stage C hyperparams (grid-search winner)
STAGEC_FINAL_RUN = "20260130_184909_7370e7a6e85f"  # reference from eval_dataset/experiments
STAGEC_W_EMBED_MAX = 0.0
STAGEC_W_EMBED = 0.7
STAGEC_CITE_WEIGHT = 0.08


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


def add_stagec_final_scores(df_in: pd.DataFrame) -> pd.DataFrame:
    """Adds `score_stageC_final` based on finalized Stage C weights.

    Required columns:
    - chapter_id
    - score_embed_max
    - score_embed_mean_top3
    - score_tfidf
    - score_cite_norm
    """
    required = [
        "chapter_id",
        "score_embed_max",
        "score_embed_mean_top3",
        "score_tfidf",
        "score_cite_norm",
    ]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C scoring: {missing}")

    df = df_in.copy()

    emb_max = pd.to_numeric(df["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(df["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    df["_emb_raw"] = float(STAGEC_W_EMBED_MAX) * emb_max + (1.0 - float(STAGEC_W_EMBED_MAX)) * emb_b

    df["_emb_n"] = minmax_by_group(df, "_emb_raw")
    df["_tf_n"] = minmax_by_group(df, "score_tfidf")
    df["_cite_n"] = minmax_by_group(df, "score_cite_norm")

    base = float(STAGEC_W_EMBED) * df["_emb_n"] + (1.0 - float(STAGEC_W_EMBED)) * df["_tf_n"]
    df["score_stageC_final"] = (1.0 - float(STAGEC_CITE_WEIGHT)) * base + float(STAGEC_CITE_WEIGHT) * df["_cite_n"]
    return df


print("Stage C finalized weights:")
print("- STAGEC_W_EMBED_MAX:", STAGEC_W_EMBED_MAX)
print("- STAGEC_W_EMBED:", STAGEC_W_EMBED)
print("- STAGEC_CITE_WEIGHT:", STAGEC_CITE_WEIGHT)


Stage C finalized weights:
- STAGEC_W_EMBED_MAX: 0.0
- STAGEC_W_EMBED: 0.7
- STAGEC_CITE_WEIGHT: 0.08


## Stage C (production): facet-union pool + embeddings + finalized scoring

Build a per-chapter candidate pool, score it, and write it to `final_pipeline/stageC/`.


In [5]:
# -----------------------------
# Stage C: pool + scoring (TF-IDF + embeddings) (cached)
# -----------------------------

import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Retrieval/scoring hyperparams (same defaults used in eval_dataset_builder.ipynb)
TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)
TOP_PER_QUERY = 250

EMBED_MODEL = "text-embedding-3-small"
MAX_CHARS_PER_EMBED = 3500
EMBED_BATCH_SIZE = 64

# Only used for score_embed_combo (not required for the finalized Stage C score)
W_EMBED_MAX = 0.70
W_EMBED_BREADTH = 0.30

EMBED_CACHE_DIR = SOURCES_WORKSPACE_DIR / ".embed_cache"
EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

STAGEC_FORCE = False

# -----------------------------
# Candidate pool utilities
# -----------------------------

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

def build_chapter_query_text(bp: dict) -> str:
    parts = []
    main = bp["main_query"]
    parts.extend([main, main])
    parts.extend(bp.get("facet_queries", []))
    parts.extend(bp.get("keywords", []))
    parts.extend(bp.get("key_concepts", []))
    return " ".join(parts)

def tfidf_scores(query_text: str, vectorizer: TfidfVectorizer, X) -> np.ndarray:
    qv = vectorizer.transform([query_text])
    return cosine_similarity(qv, X).ravel()

def facet_union_pool(bp: dict, df_in: pd.DataFrame, vectorizer: TfidfVectorizer, X, top_per_query: int) -> pd.DataFrame:
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    pool_keys = set()
    for qt in query_texts:
        qv = vectorizer.transform([qt])
        sims = cosine_similarity(qv, X).ravel()
        top_idx = np.argsort(-sims)[:top_per_query]
        pool_keys.update(df_in.iloc[top_idx]["merge_key"].astype(str).tolist())
    out = df_in[df_in["merge_key"].astype(str).isin(pool_keys)].copy()
    out = out.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)
    return out

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def hash_obj(obj: Any) -> str:
    blob = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()

def save_npz(path: Path, keys: List[str], mat: np.ndarray):
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path: Path):
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

embed_totals = {"requests": 0, "input_tokens": 0, "cost_usd": 0.0}

def embed_texts(texts: List[str], model: str, batch_size: int, max_retries: int = 6) -> np.ndarray:
    """Embeddings with retries + token/cost tracking."""
    price = MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0}).get("input", 0.0)
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)

                usage = getattr(resp, "usage", None)
                tok = int(getattr(usage, "total_tokens", 0) or getattr(usage, "prompt_tokens", 0) or 0)
                embed_totals["requests"] += 1
                embed_totals["input_tokens"] += tok
                embed_totals["cost_usd"] += (tok / 1_000_000) * price
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def score_pool_with_embeddings(bp: dict, pool: pd.DataFrame) -> pd.DataFrame:
    pool = pool.copy()
    pool["doc_text_trunc"] = (
        pool["title"].fillna("") + "\n\n" + pool["abstract"].fillna("")
    ).apply(lambda s: truncate_text(s, MAX_CHARS_PER_EMBED))

    keys = pool["merge_key"].astype(str).tolist()

    # Doc embeddings cache
    doc_hash = hash_obj({"model": EMBED_MODEL, "max_chars": MAX_CHARS_PER_EMBED, "keys": keys})[:16]
    doc_npz = EMBED_CACHE_DIR / f"eval_doc_embeds_{EMBED_MODEL}_{doc_hash}.npz"

    # Query embeddings cache
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    q_hash = hash_obj({"model": EMBED_MODEL, "queries": query_texts})[:16]
    q_json = EMBED_CACHE_DIR / f"eval_query_embeds_{EMBED_MODEL}_{q_hash}.json"

    # Docs
    if doc_npz.exists():
        cached_keys, doc_embeds = load_npz(doc_npz)
        if cached_keys != keys:
            doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
            save_npz(doc_npz, keys, doc_embeds)
    else:
        doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        save_npz(doc_npz, keys, doc_embeds)

    # Queries
    if q_json.exists():
        q_cached = json.loads(q_json.read_text(encoding="utf-8"))
        query_embeds = np.array(q_cached["embeddings"], dtype=np.float32)
    else:
        query_embeds = embed_texts(query_texts, model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        q_json.write_text(
            json.dumps({"model": EMBED_MODEL, "query_texts": query_texts, "embeddings": query_embeds.tolist()}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    docN = l2_normalize(doc_embeds)
    qN = l2_normalize(query_embeds)
    S = docN @ qN.T

    pool["score_embed_max"] = S.max(axis=1)
    pool["score_embed_mean_top3"] = np.sort(S, axis=1)[:, -3:].mean(axis=1)
    pool["score_embed_norm"] = minmax(pool["score_embed_max"].values)
    pool["score_embed_mean_top3_norm"] = minmax(pool["score_embed_mean_top3"].values)
    pool["score_embed_combo"] = W_EMBED_MAX * pool["score_embed_norm"] + W_EMBED_BREADTH * pool["score_embed_mean_top3_norm"]

    # Facet assignment (exclude the two main-query columns)
    facets = list(bp.get("facet_queries", []))
    if facets:
        facet_S = S[:, 2:2+len(facets)]
        facet_best_i = facet_S.argmax(axis=1).astype(int)
        pool["facet_best_i"] = facet_best_i
        pool["facet_best_query"] = [facets[i] for i in facet_best_i]
    else:
        pool["facet_best_i"] = -1
        pool["facet_best_query"] = ""

    return pool

# -----------------------------
# Build Stage C pool per chapter
# -----------------------------

stageC_by_chapter: Dict[str, pd.DataFrame] = {}
stageC_frames = []

def _load_latest_stagea_csv(chapter_id: str) -> pd.DataFrame:
    ch_dir = STAGEA_DIR / chapter_id
    if not ch_dir.exists():
        raise FileNotFoundError(f"No StageA dir for {chapter_id}: {ch_dir}")
    cands = sorted(ch_dir.glob('*/stageA_combined.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        raise FileNotFoundError(f"No stageA_combined.csv found under {ch_dir}")
    return pd.read_csv(cands[0])

for ch in CHAPTERS:
    cid = ch['chapter_id']
    bp_obj = blueprints[cid]
    bp = bp_obj.model_dump()

    # Load StageA corpus
    if 'stageA_by_chapter' in globals() and isinstance(stageA_by_chapter, dict) and cid in stageA_by_chapter:
        df = stageA_by_chapter[cid].copy()
    else:
        df = _load_latest_stagea_csv(cid)

    if df.empty:
        print(f"[{cid}] StageA empty; skipping")
        continue

    # Ensure required columns
    for c_req in ['merge_key', 'title', 'abstract']:
        if c_req not in df.columns:
            raise RuntimeError(f"[{cid}] StageA missing required column: {c_req}")

    df = df.copy()
    df['title'] = df['title'].fillna('')
    df['abstract'] = df['abstract'].fillna('')
    df['doc_text'] = (df['title'].astype(str) + '\n\n' + df['abstract'].astype(str)).str.strip()

    # Citation normalization (per chapter)
    cites = pd.to_numeric(df.get('citation_count_max', 0), errors='coerce').fillna(0).clip(lower=0)
    df['score_cite'] = np.log1p(cites)
    df['score_cite_norm'] = (df['score_cite'] / df['score_cite'].max()) if df['score_cite'].max() > 0 else 0.0

    # Fit TF-IDF per chapter
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
    )
    X = vectorizer.fit_transform(df['doc_text'])

    chapter_query_text = build_chapter_query_text(bp)
    df['score_tfidf'] = tfidf_scores(chapter_query_text, vectorizer=vectorizer, X=X)

    pool = facet_union_pool(bp, df_in=df, vectorizer=vectorizer, X=X, top_per_query=TOP_PER_QUERY)
    print(f"[{cid}] pool size: {len(pool)} (from StageA {len(df)})")

    pool_scored = score_pool_with_embeddings(bp, pool)
    pool_scored = add_stagec_final_scores(pool_scored)

    # Persist
    sig_obj = {
        'chapter_id': cid,
        'main_query': bp.get('main_query',''),
        'facet_queries': bp.get('facet_queries', []),
        'TOP_PER_QUERY': TOP_PER_QUERY,
        'EMBED_MODEL': EMBED_MODEL,
        'MAX_CHARS_PER_EMBED': MAX_CHARS_PER_EMBED,
        'TFIDF_MAX_FEATURES': TFIDF_MAX_FEATURES,
        'TFIDF_MIN_DF': TFIDF_MIN_DF,
        'TFIDF_NGRAM_RANGE': TFIDF_NGRAM_RANGE,
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True).encode('utf-8')).hexdigest()[:12]
    out_dir = STAGEC_DIR / cid / sig
    out_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_dir / 'stageC_pool_scored.csv'
    if (not out_csv.exists()) or STAGEC_FORCE:
        pool_scored.to_csv(out_csv, index=False, encoding='utf-8')
    stageC_by_chapter[cid] = pool_scored
    stageC_frames.append(pool_scored)

stageC_all = pd.concat(stageC_frames, axis=0).reset_index(drop=True) if stageC_frames else pd.DataFrame()
stageC_all_path = STAGEC_DIR / f"{RUN_ID}_stageC_all_pool_scored.csv"
if not stageC_all.empty:
    stageC_all.to_csv(stageC_all_path, index=False, encoding='utf-8')
    print(f"Saved Stage C combined: {stageC_all_path} | rows={len(stageC_all)}")
    print("Embedding usage summary:", embed_totals)


[zero_trust_architecture] pool size: 1313 (from StageA 2632)
Saved Stage C combined: final_pipeline\stageC\20260202_222323_stageC_all_pool_scored.csv | rows=1313
Embedding usage summary: {'requests': 22, 'input_tokens': 329778, 'cost_usd': 0.006595560000000001}


In [6]:
# -----------------------------
# Stage C.3 (finalized): LLM rerank within Stage C top-N
# -----------------------------
#
# Final decision from LOCO testing (2026-01-31):
# - Use the LLM only as a shortlist reranker to avoid full-rewrite errors.
# - Select top-N by `score_stageC_final`, then order that shortlist by LLM score.

import asyncio
import time
import random
from typing import Dict, Tuple, Literal

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm  # type: ignore
except Exception:  # pragma: no cover
    tqdm = None

STAGEC3_TOPN = 50
STAGEC3_TOPN_MAX = 150  # adaptive expansion ceiling (per chapter)
STAGEC3_TOPN_STEP = 25   # expand in chunks until enough non-excludes
STAGEC3_MIN_NON_EXCLUDE = 20  # ensure Stage D can select 20 without tail fallback

STAGEC3_MODEL = "gpt-5-nano"
STAGEC3_PROMPT_VERSION = "v2"

STAGEC3_ALPHA_LLM = 0.2  # LLM weight in Stage C.3 ordering/signal (alpha=0.2 was best in offline sweeps)

STAGEC3_CONFIDENCE_MIN = 50  # ignore low-confidence LLM judgments

STAGEC3_CONCURRENCY = 8
STAGEC3_MAX_RETRIES = 8
STAGEC3_BACKOFF_INITIAL = 1.0
STAGEC3_BACKOFF_MAX = 30.0
STAGEC3_ABSTRACT_MAX_CHARS = 2000

STAGEC3_FORCE_RERANK = False

STAGEC3_DIR = OUT_DIR / "stageC3_rerank_cache_v1"
STAGEC3_DIR.mkdir(parents=True, exist_ok=True)


class StageC3RerankOut(BaseModel):
    label: Literal["include", "maybe", "exclude"] = Field(...)
    confidence: int = Field(..., ge=0, le=100)
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)

stagec3_agent = Agent(
    name=f"StageC3 Shortlist Rerank ({STAGEC3_PROMPT_VERSION})",
    model=STAGEC3_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You validate whether a paper fits the chapter rubric.\n"
        "Return ONLY the structured output.\n\n"
        "Scoring (score is 0-100, not 0-3):\n"
        "- 90-100: excellent fit (covers multiple MUST_COVER, avoids MUST_AVOID)\n"
        "- 70-89: strong fit\n"
        "- 40-69: partial/tangential fit\n"
        "- 0-39: out of scope / wrong domain / violates MUST_AVOID\n\n"
        "label must match score: include>=70, maybe 40-69, exclude<40.\n"
        "If keywords are used in a different domain/context than the chapter, label exclude and score<=10.\n"
        "If ABSTRACT is missing, be conservative: lower confidence and avoid high scores."
    ),
    output_type=StageC3RerankOut,
)


def _stagec3_rubric_signature(bp: ChapterBlueprint) -> str:
    payload = {
        "scope_statement": bp.scope_statement,
        "must_cover": bp.must_cover,
        "must_avoid": bp.must_avoid,
        "scoring_guidance": bp.scoring_guidance,
        "prompt_version": STAGEC3_PROMPT_VERSION,
        "model": STAGEC3_MODEL,
        "abstract_max_chars": STAGEC3_ABSTRACT_MAX_CHARS,
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


def _stagec3_prompt(bp: ChapterBlueprint, title: str, abstract: str) -> str:
    scope = bp.scope_statement
    must_cover = bp.must_cover or []
    must_avoid = bp.must_avoid or []
    guidance = bp.scoring_guidance or ""

    abs_txt = (abstract or "").strip()
    if STAGEC3_ABSTRACT_MAX_CHARS and len(abs_txt) > int(STAGEC3_ABSTRACT_MAX_CHARS):
        abs_txt = abs_txt[: int(STAGEC3_ABSTRACT_MAX_CHARS)]

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("SCORING")
    lines.append("- score: integer 0-100 (not 0-3); use the full range")
    lines.append("- label must match score: include>=70, maybe 40-69, exclude<40")
    lines.append("- confidence: 0-100 certainty; use low confidence if ABSTRACT is missing")
    lines.append("- if domain/context mismatches the chapter, label exclude and score<=10")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")
    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {(title or '').strip()}")
    if abs_txt:
        lines.append(f"ABSTRACT: {abs_txt}")
    else:
        lines.append("ABSTRACT: (missing)")
    lines.append("")
    return "\n".join(lines)


def _minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)



async def stagec3_rerank_topn(
    df_in: pd.DataFrame,
    blueprints_by_chapter_id: Dict[str, ChapterBlueprint],
    *,
    topn: int = STAGEC3_TOPN,
    topn_max: int = STAGEC3_TOPN_MAX,
    topn_step: int = STAGEC3_TOPN_STEP,
    min_non_exclude: int = STAGEC3_MIN_NON_EXCLUDE,
    id_col: str = "merge_key",
) -> Tuple[pd.DataFrame, dict]:
    """Adds Stage C.3 shortlist rerank scores.

    Scientific intent:
    - We only use the LLM inside a shortlist (top-N by Stage C score).
    - In production, some chapters can have too many `exclude` labels inside top-N.
      If we then run Stage D (diversity selection), it may be forced to pick from the
      unscored tail and output irrelevant, unlabeled results.

    Therefore this function *adaptively expands* the per-chapter shortlist until we have
    at least `min_non_exclude` items labeled {include, maybe} (ignoring excludes), up to
    `topn_max`.

    Required columns in df_in:
    - chapter_id
    - title
    - abstract
    - score_stageC_final

    Returns (df_out, totals).
    """
    required = ["chapter_id", "title", "abstract", "score_stageC_final"]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C.3: {missing}")

    df = df_in.copy()

    # Ensure stable id
    if id_col not in df.columns:
        years = df["year"] if "year" in df.columns else pd.Series([""] * len(df), index=df.index)
        df[id_col] = [
            hashlib.sha1(f"{t}|{y}".encode("utf-8")).hexdigest()[:16]
            for t, y in zip(df["title"].fillna("").astype(str), years.fillna("").astype(str))
        ]

    # Output columns
    df["score_llm_rerank_v1"] = np.nan
    df["llm_notes"] = ""
    df["llm_label"] = ""
    df["llm_confidence"] = np.nan

    sem = asyncio.Semaphore(int(STAGEC3_CONCURRENCY))
    t0 = time.time()

    totals = {
        "seconds": 0.0,
        "requests": 0,
        "input_tokens": 0,
        "cached_input_tokens": 0,
        "output_tokens": 0,
        "cost_usd": 0.0,
        "cached_files": 0,
        "topn_used_by_chapter": {},
    }

    async def _run_one(chapter_id: str, doc_id: str, title: str, abstract: str) -> dict:
        bp = blueprints_by_chapter_id[chapter_id]
        sig = _stagec3_rubric_signature(bp)
        cache_dir = STAGEC3_DIR / sig / chapter_id
        cache_dir.mkdir(parents=True, exist_ok=True)
        doc_sig = hashlib.sha1(str(doc_id).encode("utf-8")).hexdigest()[:16]
        cache_path = cache_dir / f"{doc_sig}.json"

        if cache_path.exists() and not STAGEC3_FORCE_RERANK:
            payload = json.loads(cache_path.read_text(encoding="utf-8"))
            payload["_meta"] = payload.get("_meta") or {}
            payload["_meta"]["llm_cached"] = True
            payload["_meta"]["requests"] = 0
            payload["_meta"]["input_tokens"] = 0
            payload["_meta"]["cached_input_tokens"] = 0
            payload["_meta"]["output_tokens"] = 0
            payload["_meta"]["cost_usd"] = 0.0
            return payload

        prompt = _stagec3_prompt(bp, title=title, abstract=abstract)
        last_err = None
        for attempt in range(int(STAGEC3_MAX_RETRIES)):
            try:
                async with sem:
                    res = await Runner.run(stagec3_agent, prompt)
                usage = getattr(getattr(res, "context_wrapper", None), "usage", None)
                meta = cost_from_usage(usage, model=STAGEC3_MODEL) if usage is not None else {
                    "requests": 1,
                    "input_tokens": 0,
                    "cached_input_tokens": 0,
                    "output_tokens": 0,
                    "cost_usd": 0.0,
                }
                out = {
                    "label": str(res.final_output.label),
                    "confidence": int(res.final_output.confidence),
                    "score": int(res.final_output.score),
                    "notes": str(res.final_output.notes or ""),
                    "_meta": {**meta, "llm_cached": False, "rubric_sig": sig, "model": STAGEC3_MODEL},
                }
                cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
                return out
            except Exception as e:
                last_err = e
                backoff = min(float(STAGEC3_BACKOFF_MAX), float(STAGEC3_BACKOFF_INITIAL) * (2 ** attempt))
                backoff = backoff * (0.75 + 0.5 * random.random())
                await asyncio.sleep(backoff)

        raise RuntimeError(
            f"Stage C.3 rerank failed after retries for chapter_id={chapter_id} doc_id={doc_id}: {last_err}"
        )

    async def _wrapped(ix: int, chapter_id: str, doc_id: str, title: str, abstract: str) -> Tuple[int, dict]:
        payload = await _run_one(chapter_id, doc_id, title, abstract)
        return ix, payload

    # Ranked candidate indices per chapter (by Stage C score)
    ranked: Dict[str, list[int]] = {}
    desired: Dict[str, int] = {}

    for chapter_id, g in df.groupby("chapter_id"):
        cid = str(chapter_id)
        if cid not in blueprints_by_chapter_id:
            raise KeyError(f"Missing blueprint for chapter_id '{cid}'")

        sort_cols = ["score_stageC_final"]
        asc = [False]
        if id_col in g.columns:
            sort_cols.append(id_col)
            asc.append(True)

        ranked[cid] = g.sort_values(sort_cols, ascending=asc, kind="mergesort").index.to_list()
        desired[cid] = int(min(int(topn), len(ranked[cid])))

    # Iteratively score + expand
    enqueued: set[int] = set()

    def _enqueue_upto(cid: str, n: int) -> list[asyncio.Task]:
        tasks_local: list[asyncio.Task] = []
        for ix in ranked[cid][: int(n)]:
            if ix in enqueued:
                continue
            row = df.loc[ix]
            doc_id = str(row[id_col])
            title0 = str(row.get("title") or "")
            abstract0 = str(row.get("abstract") or "")
            tasks_local.append(asyncio.create_task(_wrapped(ix, cid, doc_id, title0, abstract0)))
            enqueued.add(ix)
        return tasks_local

    # Initial enqueue
    tasks: list[asyncio.Task] = []
    for cid, n in desired.items():
        tasks.extend(_enqueue_upto(cid, n))

    round_i = 0
    while tasks:
        round_i += 1
        print(
            f"[StageC3] Round {round_i}: scoring {len(tasks)} docs | model={STAGEC3_MODEL} | concurrency={STAGEC3_CONCURRENCY}"
        )

        iterator = asyncio.as_completed(tasks)
        pbar = None
        if tqdm is not None:
            pbar = tqdm(iterator, total=len(tasks), desc=f"Stage C.3 rerank ({STAGEC3_MODEL})")
            iterator = pbar

        done = 0
        for fut in iterator:
            ix, payload = await fut
            done += 1

            df.loc[ix, "score_llm_rerank_v1"] = payload.get("score")
            df.loc[ix, "llm_notes"] = payload.get("notes", "")
            df.loc[ix, "llm_label"] = payload.get("label", "")
            df.loc[ix, "llm_confidence"] = payload.get("confidence")

            m = payload.get("_meta") or {}
            totals["requests"] += int(m.get("requests", 0) or 0)
            totals["input_tokens"] += int(m.get("input_tokens", 0) or 0)
            totals["cached_input_tokens"] += int(m.get("cached_input_tokens", 0) or 0)
            totals["output_tokens"] += int(m.get("output_tokens", 0) or 0)
            totals["cost_usd"] += float(m.get("cost_usd", 0.0) or 0.0)
            totals["cached_files"] += int(bool(m.get("llm_cached", False)))

            if pbar is None and (done % 25 == 0 or done == len(tasks)):
                elapsed = float(time.time() - t0)
                print(
                    f"[StageC3] {done}/{len(tasks)} | req={totals['requests']} | cached={totals['cached_files']} | cost=${totals['cost_usd']:.4f} | sec={elapsed:.0f}"
                )

        # Decide whether to expand any chapter
        expanded_any = False
        for cid, n in list(desired.items()):
            idx = ranked[cid][: int(n)]
            labels = df.loc[idx, "llm_label"].fillna("").astype(str)
            non_ex = int((labels.ne("") & ~labels.eq("exclude")).sum())

            # store latest desired
            totals["topn_used_by_chapter"][cid] = int(n)

            if non_ex >= int(min_non_exclude):
                continue

            cap = int(min(int(topn_max), len(ranked[cid])))
            if int(n) >= cap:
                continue

            new_n = int(min(cap, int(n) + int(topn_step)))
            desired[cid] = new_n
            expanded_any = True
            print(
                f"[StageC3] Expanding chapter '{cid}': non_exclude={non_ex} < {min_non_exclude} → topn {n}→{new_n}"
            )

        if not expanded_any:
            break

        # Enqueue only newly added indices
        tasks = []
        for cid, n in desired.items():
            tasks.extend(_enqueue_upto(cid, n))

    totals["seconds"] = float(time.time() - t0)

    # Mark which docs were included in the scored shortlist (per chapter)
    df["_in_topn"] = False
    for cid, n in desired.items():
        df.loc[ranked[cid][: int(n)], "_in_topn"] = True

    # Final Stage C.3 score: only promote items that are non-exclude and pass confidence gating
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    for _, g in df[df["_in_topn"]].groupby("chapter_id"):
        llm_raw = pd.to_numeric(g["score_llm_rerank_v1"], errors="coerce").fillna(0.0)
        labels = df.loc[g.index, "llm_label"].fillna("").astype(str)
        conf = pd.to_numeric(df.loc[g.index, "llm_confidence"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        label_rank = labels.map({"exclude": 0, "maybe": 1, "include": 2}).fillna(0).astype(int).to_numpy(dtype=int)
        label_rank = np.where(conf >= float(STAGEC3_CONFIDENCE_MIN), label_rank, 0)
        llm_raw = llm_raw.where(label_rank > 0, 0.0)
        llm_n = _minmax_series(llm_raw).to_numpy(dtype=float)

        stagec = pd.to_numeric(g["score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(g["score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({
            "ix": g.index.to_list(),
            "doc_id": df.loc[g.index, id_col].fillna("").astype(str).to_list(),
            "label_rank": label_rank,
            "llm_n": llm_n,
            "stagec": stagec,
            "cite": cite,
        })
        top = top.sort_values(["label_rank", "llm_n", "stagec", "cite", "doc_id"], ascending=[False, False, False, False, True], kind="mergesort")
        boosted = top[top["label_rank"] > 0].copy()
        for r, ix in enumerate(boosted["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_final"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    print("Stage C.3 rerank totals:", {k: totals[k] for k in totals if k != 'topn_used_by_chapter'})
    print("Stage C.3 topn_used_by_chapter:", totals.get("topn_used_by_chapter"))

    return df, totals

print("Stage C.3 finalized params:")
print("- STAGEC3_TOPN:", STAGEC3_TOPN)
print("- STAGEC3_MODEL:", STAGEC3_MODEL)
print("- STAGEC3_PROMPT_VERSION:", STAGEC3_PROMPT_VERSION)


Stage C.3 finalized params:
- STAGEC3_TOPN: 50
- STAGEC3_MODEL: gpt-5-nano
- STAGEC3_PROMPT_VERSION: v2


## Stage D (finalized): MMR TF-IDF selection (diverse top-20)

Purpose: reduce redundancy in the final list while preserving (and in our benchmark improving) relevance.

Final settings (from fully adjudicated benchmark):
- `topm=100`, `lambda=0.6`, `k_select=20`
- pool ordered by `score_stageC3_topn_final`
- selection uses MMR on TF-IDF similarity, then ranks selected items by `score_stageC3_signal_v1`.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STAGED_ENABLED = True
STAGED_K_SELECT = 20
STAGED_TOPM = 100
STAGED_LAMBDA = 0.6
STAGED_TFIDF_MAX_FEATURES = 20_000


def _build_text(title: str, abstract: str) -> str:
    t = str(title or "").strip()
    a = str(abstract or "").strip()
    return (t + ". " + a).strip()


def _minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_signal_v1(
    df_in: pd.DataFrame,
    *,
    topn: int = STAGEC3_TOPN,
    stagec_col: str = "score_stageC_final",
    llm_col: str = "score_llm_rerank_v1",
    out_col: str = "score_stageC3_signal_v1",
) -> pd.DataFrame:
    """Adds a relevance signal for Stage D.

    - For items inside Stage C top-N: `1 + minmax(llm_score)` (label/confidence gated)
    - For the tail: `score_stageC_final`
    """
    df = df_in.copy()
    df[out_col] = pd.to_numeric(df[stagec_col], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        if "_in_topn" in g.columns:
            idx = g[g["_in_topn"].fillna(False)].index
            if len(idx) == 0:
                idx = g.sort_values(stagec_col, ascending=False, kind="mergesort").head(int(topn)).index
        else:
            idx = g.sort_values(stagec_col, ascending=False, kind="mergesort").head(int(topn)).index
        llm_raw = pd.to_numeric(df.loc[idx, llm_col], errors="coerce").fillna(0.0)
        gate = pd.Series(True, index=idx)
        if "llm_label" in df.columns:
            labels = df.loc[idx, "llm_label"].fillna("").astype(str)
            gate = gate & ~labels.eq("exclude")
        if "llm_confidence" in df.columns:
            conf = pd.to_numeric(df.loc[idx, "llm_confidence"], errors="coerce").fillna(0.0)
            conf_min = float(globals().get("STAGEC3_CONFIDENCE_MIN", 0) or 0)
            if conf_min > 0:
                gate = gate & (conf >= conf_min)

        valid_idx = idx[gate.to_numpy(dtype=bool)]
        if len(valid_idx):
            llm_n_valid = _minmax_series(llm_raw.loc[valid_idx])
            df.loc[valid_idx, out_col] = 1.0 + llm_n_valid
    return df


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))
    return chosen


def add_stageD_mmr_tfidf_v2(
    df_in: pd.DataFrame,
    *,
    baseline_col: str = "score_stageC3_topn_final",
    relevance_col: str = "score_stageC3_signal_v1",
    title_col: str = "title",
    abstract_col: str = "abstract",
    id_col: str = "merge_key",
    topm: int = STAGED_TOPM,
    k_select: int = STAGED_K_SELECT,
    lam: float = STAGED_LAMBDA,
    out_col: str = "score_stageD_final",
) -> pd.DataFrame:
    """Stage D: select a diverse top-K using TF-IDF MMR.

    Output behavior:
    - Selected docs get scores `3.0 - r*1e-6` (strict ordering).
    - All other docs keep `baseline_col` scores.
    """
    df = df_in.copy()
    required = ["chapter_id", baseline_col, relevance_col, title_col, abstract_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage D: {missing}")

    df[out_col] = pd.to_numeric(df[baseline_col], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        g_sorted = g.sort_values([baseline_col, id_col], ascending=[False, True], kind="mergesort")
        pool = g_sorted.copy()
        # Prefer selecting only from the docs that Stage C.3 actually scored (avoids unlabeled tail docs).
        if "_in_topn" in pool.columns:
            pool = pool[pool["_in_topn"].fillna(False)].copy()
        if 'llm_label' in pool.columns:
            pool = pool[pool['llm_label'].fillna('').astype(str).ne('')].copy()
            pool = pool[~pool['llm_label'].fillna('').astype(str).eq('exclude')].copy()
        if int(topm) > 0 and len(pool) > int(topm):
            pool = pool.sort_values([baseline_col, id_col], ascending=[False, True], kind="mergesort").head(int(topm)).copy()
        if len(pool) == 0:
            print(f"[StageD] Warning: empty pool for {cid}; skipping Stage D for this chapter.")
            continue
        if len(pool) < int(k_select):
            print(f"[StageD] Warning: pool<{k_select} for {cid}: pool={len(pool)}. Consider increasing Stage C.3 topn/topn_max.")
        texts = [_build_text(t, a) for t, a in zip(pool[title_col].fillna(""), pool[abstract_col].fillna(""))]
        tfidf = TfidfVectorizer(max_features=int(STAGED_TFIDF_MAX_FEATURES))
        X = tfidf.fit_transform(texts)
        S = cosine_similarity(X)

        rel = pd.to_numeric(pool[relevance_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        chosen_local = mmr_select(rel, S, k=int(k_select), lam=float(lam))
        chosen_idx = pool.iloc[chosen_local].index.tolist()

        chosen_sorted = (
            pool.loc[chosen_idx]
            .sort_values([relevance_col, baseline_col, id_col], ascending=[False, False, True], kind="mergesort")
            .index.tolist()
        )
        for r, ix in enumerate(chosen_sorted):
            df.loc[ix, out_col] = 3.0 - (r * 1e-6)

    return df


print("Stage D finalized params:")
print("- enabled:", STAGED_ENABLED)
print("- k_select:", STAGED_K_SELECT)
print("- topm:", STAGED_TOPM)
print("- lambda:", STAGED_LAMBDA)


Stage D finalized params:
- enabled: True
- k_select: 20
- topm: 100
- lambda: 0.6


## Final run (production)

Runs Stage C.3 (LLM rerank within top‑N) and Stage D (MMR TF‑IDF) and writes outputs to `final_pipeline/results/`.
After you `Run All`, paste the printed JSON summary back here.


In [8]:

# -----------------------------
# Final run: Stage C.3 + Stage D + exports
# -----------------------------

import json
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover
    def display(x):  # type: ignore
        print(x)


def _latest(path_glob):
    cands = sorted(path_glob, key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None


# Load Stage C pool if needed
if "stageC_all" not in globals() or not isinstance(stageC_all, pd.DataFrame) or stageC_all.empty:
    p = _latest(list(STAGEC_DIR.glob("*_stageC_all_pool_scored.csv")))
    if p is None:
        raise FileNotFoundError("No Stage C combined CSV found under final_pipeline/stageC/. Run the Stage C cell first.")
    stageC_all = pd.read_csv(p)
    print("Loaded Stage C combined:", p, "| rows=", len(stageC_all))
else:
    print("Using in-memory stageC_all | rows=", len(stageC_all))


# Stage C.3 (API calls; cached to disk)
stageC3_df, stageC3_totals = await stagec3_rerank_topn(
    stageC_all,
    blueprints_by_chapter_id=blueprints,
    min_non_exclude=int(STAGED_K_SELECT),
)


# Stage D (offline)
final_score_col = "score_stageC3_topn_final"
if STAGED_ENABLED:
    stageC3_df = add_stagec3_signal_v1(stageC3_df)
    stageC3_df = add_stageD_mmr_tfidf_v2(stageC3_df)
    final_score_col = "score_stageD_final"


# Save full scored output
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
full_scored_csv = RESULTS_DIR / f"{RUN_ID}_full_scored.csv"
stageC3_df.to_csv(full_scored_csv, index=False, encoding="utf-8")
print("Saved full scored CSV:", full_scored_csv)


# Save per-chapter top20
top20_paths = {}
cols_show = [
    "chapter_id",
    "title",
    "year",
    "venue",
    "doi_norm",
    "sources",
    "score_stageC_final",
    "score_llm_rerank_v1",
    "llm_label",
    "llm_confidence",
    "llm_notes",
    final_score_col,
    "merge_key",
]

for cid, g in stageC3_df.groupby("chapter_id"):
    out_csv = RESULTS_DIR / f"{RUN_ID}_{cid}_top20.csv"
    top = g.sort_values(final_score_col, ascending=False, kind="mergesort").copy()
    if "llm_label" in top.columns:
        top = top[~top["llm_label"].fillna("").astype(str).eq("exclude")].copy()
    top = top.head(20).copy()
    keep = [c for c in cols_show if c in top.columns]
    top[keep].to_csv(out_csv, index=False, encoding="utf-8")
    top20_paths[str(cid)] = str(out_csv)
    print(f"[{cid}] saved top20: {out_csv}")
    display(top[keep].head(10))


# Build a concise run summary for copy/paste
stagea_paths = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    # The Stage A cell defines query_list_for_chapter; reuse it to compute the sig.
    try:
        queries = query_list_for_chapter(bp)
    except Exception:
        queries = [bp.main_query] + list(bp.facet_queries or [])
    sig_obj = {
        "queries": queries,
        "OA_MAX_WORKS_PER_QUERY": globals().get("OA_MAX_WORKS_PER_QUERY"),
        "S2_LIMIT": globals().get("S2_LIMIT"),
        "S2_MAX_PAGES_PER_QUERY": globals().get("S2_MAX_PAGES_PER_QUERY"),
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    stagea_paths[cid] = str((STAGEA_DIR / cid / sig / "stageA_combined.csv").resolve())

summary = {
    "run_id": RUN_ID,
    "chapters": [c["chapter_id"] for c in CHAPTERS],
    "stageB": {"variant": STAGEB_FINAL_VARIANT, "model": BLUEPRINT_MODEL},
    "stageA": {"paths": stagea_paths},
    "stageC": {
        "weights": {
            "w_embed_max": STAGEC_W_EMBED_MAX,
            "w_embed": STAGEC_W_EMBED,
            "cite_weight": STAGEC_CITE_WEIGHT,
        },
        "embed_totals": globals().get("embed_totals", {}),
    },
    "stageC3": {"topn": STAGEC3_TOPN, "model": STAGEC3_MODEL, "totals": stageC3_totals},
    "stageD": {"enabled": STAGED_ENABLED, "topm": STAGED_TOPM, "k_select": STAGED_K_SELECT, "lambda": STAGED_LAMBDA},
    "outputs": {
        "full_scored_csv": str(full_scored_csv),
        "top20_csvs": top20_paths,
        "final_score_col": final_score_col,
    },
}

summary_path = RESULTS_DIR / f"{RUN_ID}_run_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


# -----------------------------
# QA summary (sanity checks)
# -----------------------------
print('\nQA summary (top-20):')
for cid, g in stageC3_df.groupby('chapter_id'):
    top = g.sort_values(final_score_col, ascending=False, kind='mergesort').head(20)
    if 'llm_label' in top.columns:
        counts = top['llm_label'].fillna('').astype(str).value_counts().to_dict()
        print(f'- {cid}: llm_label counts in top20 = {counts}')
        if counts.get('exclude', 0) > 0:
            print(f'  WARNING: excludes present in top20 for {cid}')
        if counts.get('', 0) > 0:
            print(f'  WARNING: blank llm_label present in top20 for {cid} (likely unscored tail)')

print("\n=== COPY/PASTE THIS JSON SUMMARY BACK TO CODEX ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved summary file:", summary_path)


Using in-memory stageC_all | rows= 1313
[StageC3] Round 1: scoring 50 docs | model=gpt-5-nano | concurrency=8


Stage C.3 rerank (gpt-5-nano):   0%|          | 0/50 [00:00<?, ?it/s]

Stage C.3 rerank totals: {'seconds': 105.9945809841156, 'requests': 50, 'input_tokens': 39062, 'cached_input_tokens': 0, 'output_tokens': 58841, 'cost_usd': 0.0254895, 'cached_files': 0}
Stage C.3 topn_used_by_chapter: {'zero_trust_architecture': 50}
Saved full scored CSV: final_pipeline\results\20260202_222323_full_scored.csv
[zero_trust_architecture] saved top20: final_pipeline\results\20260202_222323_zero_trust_architecture_top20.csv


,chapter_id,title,year,venue,doi_norm,sources,score_stageC_final,score_llm_rerank_v1,llm_label,llm_confidence,llm_notes,score_stageD_final,merge_key
1240,zero_trust_architecture,Practical Zero-Trust Deployment for Hybrid Wor...,2025.0,Innovative Research Thoughts,10.36676/irt.v11.i4.1710,semantic_scholar,0.796715,92.0,include,90.0,Strong fit: comprehensive ZTA scope with refer...,3.000000,doi:10.36676/irt.v11.i4.1710
1233,zero_trust_architecture,Zero Trust Architecture and Business Risk Alig...,2024.0,International Journal of Scientific Research a...,https://doi.org/10.18535/ijsrm/v12i10.ec13,openalex,0.786164,88.0,include,85.0,"Strong alignment with ZTA scope: definitions, ...",2.999999,doi:https://doi.org/10.18535/ijsrm/v12i10.ec13
787,zero_trust_architecture,Zero Trust Security: Reimagining Cyber Defense...,2022.0,International Journal of Scientific Research a...,10.18535/ijsrm/v10i4.ec11,semantic_scholar,0.773963,88.0,include,85.0,"Strong fit: covers core ZTA principles, core c...",2.999998,doi:10.18535/ijsrm/v10i4.ec11
1278,zero_trust_architecture,Best Practices for Implementing Zero Trust in ...,2025.0,European Journal of Computer Science and Infor...,https://doi.org/10.37745/ejcsit.2013/vol13n339...,openalex,0.741245,88.0,include,80.0,Strong fit for ZTA chapter: covers verificatio...,2.999997,doi:https://doi.org/10.37745/ejcsit.2013/vol13...
956,zero_trust_architecture,Zero trust at scale: Security architecture for...,2025.0,World Journal of Advanced Research and Reviews,10.30574/wjarr.2025.26.2.1939,semantic_scholar,0.859820,86.0,include,90.0,Strong ZTA architectural fit: identity-centric...,2.999996,doi:10.30574/wjarr.2025.26.2.1939
1259,zero_trust_architecture,Evaluating the Effectiveness of Zero-Trust Arc...,2025.0,2025 International Conference on Sustainabilit...,10.1109/icsit65336.2025.11294850,semantic_scholar,0.728345,85.0,include,88.0,"Strong fit: defines ZTA principles (verify, le...",2.999995,doi:10.1109/icsit65336.2025.11294850
1186,zero_trust_architecture,Zero Trust Architecture: A Comprehensive Frame...,2025.0,International Journal of Advanced Research in ...,10.48175/ijarsct-24449,semantic_scholar,0.772649,84.0,include,85.0,Strong alignment with ZTA scope: core principl...,2.999994,doi:10.48175/ijarsct-24449
1010,zero_trust_architecture,Zero trust architecture in modern computer net...,2020.0,World Journal of Advanced Research and Reviews,10.30574/wjarr.2020.7.3.0280,semantic_scholar,0.815182,82.0,include,85.0,Strong fit for ZTA scope and core principles; ...,2.999993,doi:10.30574/wjarr.2020.7.3.0280
973,zero_trust_architecture,The Zero Trust Paradigm: Revolutionizing Netwo...,2025.0,2025 International Conference on Networks and ...,10.1109/netcrypt65877.2025.11102576,semantic_scholar,0.809223,82.0,include,78.0,"Strong fit: defines Zero Trust concepts, core ...",2.999992,doi:10.1109/netcrypt65877.2025.11102576
1098,zero_trust_architecture,Beyond the Perimeter: Reimagining Cloud and Hy...,2025.0,International Journal for Research in Applied ...,10.22214/ijraset.2025.75617,semantic_scholar,0.783689,82.0,include,80.0,Strong fit for ZTA scope in enterprise cloud/h...,2.999991,doi:10.22214/ijraset.2025.75617



QA summary (top-20):
- zero_trust_architecture: llm_label counts in top20 = {'include': 20}

=== COPY/PASTE THIS JSON SUMMARY BACK TO CODEX ===
{
  "run_id": "20260202_222323",
  "chapters": [
    "zero_trust_architecture"
  ],
  "stageB": {
    "variant": "coverage_v1",
    "model": "gpt-5-mini"
  },
  "stageA": {
    "paths": {
      "zero_trust_architecture": "E:\\Datein\\Coding\\instantpaper\\final_pipeline\\stageA\\zero_trust_architecture\\fa4c38fa4185\\stageA_combined.csv"
    }
  },
  "stageC": {
    "weights": {
      "w_embed_max": 0.0,
      "w_embed": 0.7,
      "cite_weight": 0.08
    },
    "embed_totals": {
      "requests": 22,
      "input_tokens": 329778,
      "cost_usd": 0.006595560000000001
    }
  },
  "stageC3": {
    "topn": 50,
    "model": "gpt-5-nano",
    "totals": {
      "seconds": 105.9945809841156,
      "requests": 50,
      "input_tokens": 39062,
      "cached_input_tokens": 0,
      "output_tokens": 58841,
      "cost_usd": 0.0254895,
      "cached_fi